<a href="https://colab.research.google.com/github/SlightlyOffset/AI-companion/blob/main/colab_bridge/XTTS_Bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ AI Companion: XTTS v2 Remote Voice Bridge

### 🛠️ Setup Instructions
1. **GPU**: `Runtime` > `Change runtime type` > **T4 GPU**.
2. **Ngrok**: Add `NGROK_TOKEN` to **Secrets** (key icon) and enable "Notebook access".
3. **Run All**: `Ctrl + F9`.

**Note**: This version uses PyTorch 2.5.1 to ensure compatibility with XTTS v2.

In [ ]:
# @title 1. Prepare Environment
import os

print("Installing Python 3.10...")
!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev > /dev/null

print("Creating virtual environment...")
if not os.path.exists("xtts_env"):
    !python3.10 -m venv xtts_env

print("Installing XTTS dependencies (3-5 minutes)...")
!./xtts_env/bin/pip install -q -U pip
# Downgrade torch to 2.5.1 to avoid the PyTorch 2.6 weights_only security error
print("Installing stable PyTorch 2.5.1...")
!./xtts_env/bin/pip install -q fastapi uvicorn torch==2.5.1 torchaudio==2.5.1 transformers==4.33.0 TTS python-multipart

print("\n✅ Environment Ready!")

In [ ]:
# @title 2. Create Bridge Script
with open("bridge_server.py", "w") as f:
    f.write("""
import torch
import os
import shutil
import uuid
from typing import List
from TTS.api import TTS
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import uvicorn
import io

SPEAKERS_DIR = "speakers"
os.makedirs(SPEAKERS_DIR, exist_ok=True)

print("-> Loading XTTS v2 model... (This may take a minute)")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Model Loaded.")

app = FastAPI()

@app.get("/check_speaker/{speaker_id}")
async def check_speaker(speaker_id: str):
    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    if os.path.exists(speaker_path) and os.listdir(speaker_path):
        return {"exists": True, "speaker_id": speaker_id}
    return {"exists": False, "speaker_id": speaker_id}

@app.post("/upload_speaker")
async def upload_speaker(
    speaker_id: str = Form(...),
    files: List[UploadFile] = File(...)
):
    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    os.makedirs(speaker_path, exist_ok=True)
    
    # Clear existing files for this speaker
    for f in os.listdir(speaker_path):
        try: os.remove(os.path.join(speaker_path, f))
        except: pass
        
    for upload in files:
        file_path = os.path.join(speaker_path, upload.filename)
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(upload.file, buffer)
            
    print(f"✅ Speaker '{speaker_id}' registered with {len(files)} samples.")
    return {"status": "success", "speaker_id": speaker_id}

@app.post("/generate_tts")
async def generate_tts_endpoint(
    text: str = Form(...),
    language: str = Form("en"),
    speaker_id: str = Form(None)
):
    if not speaker_id:
        raise HTTPException(status_code=400, detail="speaker_id is required")
        
    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    if not os.path.exists(speaker_path):
        raise HTTPException(status_code=404, detail=f"Speaker '{speaker_id}' not found.")
        
    speaker_wavs = [os.path.join(speaker_path, f) for f in os.listdir(speaker_path) if f.endswith(".wav")]
    
    # Streaming generator
    def stream_audio():
        # inference_stream yields audio chunks as tensors
        chunks = tts_model.inference_stream(
            text=text,
            language=language,
            speaker_wav=speaker_wavs,
            stream_chunk_size=20
        )
        for chunk in chunks:
            # Convert tensor to raw bytes (PCM 16-bit, 24kHz)
            yield chunk.cpu().numpy().tobytes()

    return StreamingResponse(stream_audio(), media_type="audio/l16; rate=24000")
""")
print("✅ Bridge script created.")

In [ ]:
# @title 3. Start XTTS Bridge
import sys
import subprocess
import time

try:
    from pyngrok import ngrok
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyngrok"])
    from pyngrok import ngrok

from google.colab import userdata
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
except:
    print("❌ ERROR: NGROK_TOKEN not found in Secrets!")

ngrok.kill()
public_url = ngrok.connect(8001).public_url

print("="*50)
print(f"\n🚀 XTTS BRIDGE ONLINE!\n")
print(f"URL: {public_url}\n")
print("="*50)

# Run server
!./xtts_env/bin/uvicorn bridge_server:app --host 0.0.0.0 --port 8001 --log-level error